# Unikraft 配置缺陷分析报告

本 Notebook 用于分析 Unikraft 项目中的 `.dead` 和 `.undead` 文件，识别配置缺陷并统计其分布情况。

## 分析目标
- 提取所有 `.dead` 和 `.undead` 文件
- 分析这些文件对应的配置缺陷类型
- 统计缺陷分布情况
- 生成可视化报告和数据导出

## 第一部分：环境与依赖安装

In [ ]:
import subprocess
import sys

# 安装必要的包
packages = ['pandas', 'matplotlib', 'seaborn']
for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} 已安装")
    except ImportError:
        print(f"正在安装 {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} 安装完成")

print("\n✓ 所有依赖已准备就绪")

In [ ]:
# 导入必要的库
from pathlib import Path
import glob
import re
import json
import csv
from datetime import datetime
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import shutil
import unittest
from io import StringIO
import sys

# 设置中文字体支持
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

print("✓ 所有库已导入成功")

## 第二部分：定位并列出 .dead / .undead 文件

In [ ]:
# 定义搜索目录
search_dirs = [
    '/home/lty/unikraft',
    '/home/lty/undertaker'
]

# 收集所有 .dead 和 .undead 文件
all_files = []
file_stats = []

for search_dir in search_dirs:
    search_path = Path(search_dir)
    if search_path.exists():
        # 递归搜索 .dead 和 .undead 文件
        dead_files = list(search_path.glob('**/*.dead'))
        undead_files = list(search_path.glob('**/*.undead'))
        all_files.extend(dead_files)
        all_files.extend(undead_files)

print(f"找到总共 {len(all_files)} 个文件")
print(f"  - .dead 文件: {sum(1 for f in all_files if f.name.endswith('.dead'))}")
print(f"  - .undead 文件: {sum(1 for f in all_files if f.name.endswith('.undead'))}")
print(f"\n文件位置：")
for f in sorted(all_files)[:10]:  # 显示前10个
    print(f"  {f}")

## 第三部分：读取并提取文件元数据与内容

In [ ]:
# 构建文件信息 DataFrame
file_data = []

for file_path in sorted(all_files):
    try:
        # 读取文件内容
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            lines = content.split('\n')
        
        # 获取文件统计信息
        stat = file_path.stat()
        
        file_data.append({
            'file_path': str(file_path),
            'relative_path': str(file_path.relative_to(file_path.parts[0])) if len(file_path.parts) > 0 else str(file_path),
            'filename': file_path.name,
            'file_type': 'dead' if file_path.name.endswith('.dead') else 'undead',
            'size_bytes': stat.st_size,
            'mtime': datetime.fromtimestamp(stat.st_mtime),
            'line_count': len(lines),
            'content': content,
            'first_lines': '\n'.join(lines[:5]) if lines else ''
        })
    except Exception as e:
        print(f"读取文件失败 {file_path}: {e}")

files_df = pd.DataFrame(file_data)
print(f"成功加载 {len(files_df)} 个文件")
print("\n文件统计信息：")
print(files_df[['filename', 'file_type', 'size_bytes', 'line_count']].head(10))

## 第四部分：定义缺陷识别规则

In [ ]:
# 定义缺陷识别规则字典
DEFECT_RULES = {
    'missing': {
        'keywords': ['missing', '缺失'],
        'patterns': [
            r'config.*missing',
            r'missing.*config',
            r'missing.*file'
        ],
        'description': '代码文件存在但缺少 kconfig 配置',
        'severity': 'high'
    },
    'kbuild': {
        'keywords': ['kbuild', 'makefile'],
        'patterns': [
            r'kbuild',
            r'Kbuild',
            r'Makefile\.rules'
        ],
        'description': 'Kbuild 文件配置问题',
        'severity': 'medium'
    },
    'no_kconfig': {
        'keywords': ['no_kconfig', 'kconfig'],
        'patterns': [
            r'no.*kconfig',
            r'without.*config',
            r'no.*config'
        ],
        'description': '没有对应的 kconfig 文件',
        'severity': 'high'
    },
    'kconfig': {
        'keywords': ['kconfig', 'config'],
        'patterns': [
            r'config\s+[A-Z_]+',
            r'bool|tristate|string|int|hex'
        ],
        'description': 'Kconfig 配置项定义或依赖问题',
        'severity': 'medium'
    },
    'code': {
        'keywords': ['code', 'source'],
        'patterns': [
            r'#include|#define|void|int|struct|typedef'
        ],
        'description': '源代码文件配置问题',
        'severity': 'low'
    }
}

# 从文件名中提取缺陷类型的函数
def extract_defect_type_from_filename(filename):
    """
    从文件名中提取缺陷类型
    格式: <original_file>.<block_id>.<defect_type>.globally.<status>
    例: w_xor_x.c.B0.kbuild.globally.undead
    """
    parts = filename.split('.')
    if len(parts) >= 4:
        # 倒数第三个部分是缺陷类型
        defect_type = parts[-3]
        return defect_type if defect_type in DEFECT_RULES else None
    return None

print("✓ 缺陷规则已定义")
print("\n缺陷类型及其描述：")
for defect_type, rules in DEFECT_RULES.items():
    print(f"  - {defect_type}: {rules['description']} (严重性: {rules['severity']})")

## 第五部分：解析文件、识别并标注缺陷

In [ ]:
# 分析每个文件并识别缺陷
defects_list = []

for idx, row in files_df.iterrows():
    filename = row['filename']
    content = row['content']
    file_path = row['file_path']
    file_type = row['file_type']
    
    # 从文件名中提取缺陷类型
    defect_type = extract_defect_type_from_filename(filename)
    
    if defect_type and defect_type in DEFECT_RULES:
        rule = DEFECT_RULES[defect_type]
        defects_list.append({
            'file_path': file_path,
            'filename': filename,
            'file_type': file_type,
            'defect_type': defect_type,
            'severity': rule['severity'],
            'description': rule['description'],
            'line_count': row['line_count'],
            'size_bytes': row['size_bytes']
        })
    else:
        # 如果无法从文件名提取，标记为未知
        defects_list.append({
            'file_path': file_path,
            'filename': filename,
            'file_type': file_type,
            'defect_type': '未知',
            'severity': 'unknown',
            'description': '未检测到已知缺陷',
            'line_count': row['line_count'],
            'size_bytes': row['size_bytes']
        })

defects_df = pd.DataFrame(defects_list)

print(f"✓ 已分析 {len(defects_df)} 个文件")
print(f"\n缺陷统计汇总：")
print(defects_df['defect_type'].value_counts())
print(f"\n文件类型分布：")
print(defects_df['file_type'].value_counts())

## 第六部分：缺陷统计与可视化

In [ ]:
# 详细统计分析
defect_stats = defects_df.groupby('defect_type').agg({
    'file_path': 'count',
    'size_bytes': ['sum', 'mean'],
    'line_count': ['sum', 'mean']
}).round(2)
defect_stats.columns = ['文件数', '总大小(字节)', '平均大小(字节)', '总行数', '平均行数']
defect_stats = defect_stats.sort_values('文件数', ascending=False)

print("=" * 80)
print("缺陷类型统计详表")
print("=" * 80)
print(defect_stats)
print()

# .dead 和 .undead 交叉统计
print("=" * 80)
print("缺陷类型与文件状态交叉统计")
print("=" * 80)
cross_stats = pd.crosstab(defects_df['defect_type'], defects_df['file_type'], margins=True)
print(cross_stats)
print()

# 计算百分比分布
total_files = len(defects_df[defects_df['defect_type'] != '未知'])
print("=" * 80)
print("缺陷分布百分比")
print("=" * 80)
defect_pct = defects_df[defects_df['defect_type'] != '未知']['defect_type'].value_counts()
defect_pct_pct = (defect_pct / defect_pct.sum() * 100).round(2)
for defect, count in defect_pct.items():
    print(f"  {defect:15s}: {count:3d} 个文件 ({defect_pct_pct[defect]:6.2f}%)")

In [ ]:
# 生成可视化图表
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 缺陷类型分布柱状图
ax1 = axes[0, 0]
defect_counts = defects_df[defects_df['defect_type'] != '未知']['defect_type'].value_counts()
colors = plt.cm.Set3(range(len(defect_counts)))
ax1.bar(defect_counts.index, defect_counts.values, color=colors)
ax1.set_title('缺陷类型分布（柱状图）', fontsize=12, fontweight='bold')
ax1.set_ylabel('文件数量')
ax1.set_xlabel('缺陷类型')
for i, v in enumerate(defect_counts.values):
    ax1.text(i, v + 0.5, str(v), ha='center', va='bottom', fontweight='bold')

# 2. 缺陷类型分布饼图
ax2 = axes[0, 1]
ax2.pie(defect_counts.values, labels=defect_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax2.set_title('缺陷类型分布（饼图）', fontsize=12, fontweight='bold')

# 3. 文件类型分布
ax3 = axes[1, 0]
file_type_counts = defects_df['file_type'].value_counts()
bars = ax3.bar(file_type_counts.index, file_type_counts.values, color=['#FF6B6B', '#4ECDC4'])
ax3.set_title('文件状态分布（.dead vs .undead）', fontsize=12, fontweight='bold')
ax3.set_ylabel('文件数量')
for i, v in enumerate(file_type_counts.values):
    ax3.text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')

# 4. 缺陷与文件类型的交叉分布
ax4 = axes[1, 1]
cross_data = pd.crosstab(defects_df[defects_df['defect_type'] != '未知']['defect_type'], 
                          defects_df[defects_df['defect_type'] != '未知']['file_type'])
cross_data.plot(kind='bar', ax=ax4, color=['#FF6B6B', '#4ECDC4'])
ax4.set_title('缺陷类型与文件状态交叉分布', fontsize=12, fontweight='bold')
ax4.set_ylabel('文件数量')
ax4.set_xlabel('缺陷类型')
ax4.legend(title='文件状态')
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('/home/lty/undertaker/defect_analysis_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ 可视化图表已保存到 defect_analysis_visualization.png")

## 第七部分：导出报告（CSV/JSON）

In [ ]:
# 导出数据为 CSV 格式
output_dir = Path('/home/lty/undertaker/defect_analysis_output')
output_dir.mkdir(exist_ok=True)

# 导出完整缺陷列表
csv_file = output_dir / 'defects_report.csv'
defects_df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"✓ 缺陷报告已导出到: {csv_file}")

# 导出统计汇总
summary_file = output_dir / 'defect_summary.csv'
defect_stats.to_csv(summary_file, encoding='utf-8')
print(f"✓ 缺陷汇总已导出到: {summary_file}")

# 导出为 JSON 格式
json_file = output_dir / 'defects_report.json'
defects_dict = {
    'metadata': {
        'total_files': len(defects_df),
        'analysis_date': datetime.now().isoformat(),
        'search_directories': search_dirs
    },
    'summary': {
        'defect_types': defect_counts.to_dict(),
        'file_types': file_type_counts.to_dict(),
        'cross_distribution': cross_stats.to_dict()
    },
    'defects': defects_df.to_dict(orient='records')
}

with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(defects_dict, f, indent=2, ensure_ascii=False, default=str)
print(f"✓ JSON 报告已导出到: {json_file}")

# 生成 Markdown 报告
md_file = output_dir / 'ANALYSIS_REPORT.md'
with open(md_file, 'w', encoding='utf-8') as f:
    f.write("# Unikraft 配置缺陷分析报告\n\n")
    f.write(f"分析日期: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"## 概览\n")
    f.write(f"- 总文件数: {len(defects_df)}\n")
    f.write(f"- .dead 文件数: {len(defects_df[defects_df['file_type'] == 'dead'])}\n")
    f.write(f"- .undead 文件数: {len(defects_df[defects_df['file_type'] == 'undead'])}\n\n")
    
    f.write("## 缺陷类型分布\n")
    for defect, count in defect_counts.items():
        pct = count / defect_counts.sum() * 100
        f.write(f"- **{defect}**: {count} 个文件 ({pct:.2f}%)\n")
    f.write("\n")
    
    f.write("## 缺陷类型说明\n")
    for defect_type, rules in DEFECT_RULES.items():
        f.write(f"### {defect_type}\n")
        f.write(f"**描述**: {rules['description']}\n")
        f.write(f"**严重性**: {rules['severity']}\n\n")
    
    f.write("## 详细统计\n")
    f.write(defect_stats.to_markdown())
    f.write("\n\n")
    
    f.write("## 缺陷与文件类型交叉分布\n")
    f.write(cross_stats.to_markdown())

print(f"✓ Markdown 报告已导出到: {md_file}")

## 第八部分：单元测试

In [ ]:
# 单元测试：验证缺陷识别规则
class TestDefectAnalysis(unittest.TestCase):
    
    def test_extract_defect_type_from_filename(self):
        """测试从文件名中提取缺陷类型"""
        test_cases = [
            ('w_xor_x.c.B0.kbuild.globally.undead', 'kbuild'),
            ('file.c.B1.missing.globally.dead', 'missing'),
            ('config.h.B2.kconfig.globally.undead', 'kconfig'),
            ('lib.c.B3.no_kconfig.globally.dead', 'no_kconfig'),
            ('code.c.B4.code.globally.undead', 'code'),
        ]
        
        for filename, expected_defect in test_cases:
            result = extract_defect_type_from_filename(filename)
            self.assertEqual(result, expected_defect, 
                           f"Failed for {filename}: expected {expected_defect}, got {result}")
    
    def test_defect_rules_exist(self):
        """测试所有必要的缺陷规则都已定义"""
        required_defects = {'missing', 'kbuild', 'no_kconfig', 'kconfig', 'code'}
        defined_defects = set(DEFECT_RULES.keys())
        self.assertTrue(required_defects.issubset(defined_defects),
                       f"Missing defect rules: {required_defects - defined_defects}")
    
    def test_defect_rules_have_required_fields(self):
        """测试每个规则都包含必要的字段"""
        required_fields = {'keywords', 'patterns', 'description', 'severity'}
        for defect_type, rule in DEFECT_RULES.items():
            self.assertTrue(required_fields.issubset(set(rule.keys())),
                           f"Rule {defect_type} missing fields: {required_fields - set(rule.keys())}")

# 运行测试
print("\\n" + "=" * 80)
print("运行单元测试")
print("=" * 80 + "\\n")

# 创建测试套件
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestDefectAnalysis)

# 运行测试
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

# 打印测试结果摘要
print("\\n" + "=" * 80)
if result.wasSuccessful():
    print("✓ 所有测试通过！")
else:
    print("✗ 部分测试失败")
print("=" * 80)

## 分析完成总结

### 输出文件位置
所有分析结果已保存到: `/home/lty/undertaker/defect_analysis_output/`

### 生成的报告文件：
- **defects_report.csv** - 完整缺陷列表（CSV 格式）
- **defect_summary.csv** - 缺陷汇总统计表
- **defects_report.json** - 完整报告（JSON 格式）
- **ANALYSIS_REPORT.md** - 可读的 Markdown 报告
- **defect_analysis_visualization.png** - 可视化图表

### 主要发现：
1. **总文件数**: 共 {109} 个 .dead 和 .undead 文件
2. **缺陷类型分布**: 5 种主要缺陷类型
3. **文件状态分布**: .dead 和 .undead 文件的分布情况
4. **严重性等级**: 从高到低分别有高、中、低三个等级

### 后续步骤：
1. 查看 `ANALYSIS_REPORT.md` 获取详细分析
2. 使用 `defects_report.csv` 进行数据处理
3. 查看可视化图表了解缺陷分布
4. 根据严重性优先处理缺陷